# FER-CNN — all experiments (Google Colab, GPU)

Clones the repo, downloads FER-2013 and FANE from Kaggle, then runs every
experiment of the study and archives each one in its own Drive folder.

| folder on Drive | what it contains |
|---|---|
| `eda/` | dataset figures and class counts |
| `custom_cnn/` | ResNet-style CNN trained from scratch |
| `xception_1phase/` | Xception, tail unfrozen from the first epoch |
| `xception_2phase/` | Xception, head trained first and tail unfrozen after |
| `final/` | both models evaluated together, comparison tables and plots |

The two Xception variants exist to check whether the warm-up phase helps. The
winner is picked on validation loss only, never on the test sets.

## 1. Repository and dependencies

In [ ]:
REPO_URL = 'https://github.com/xydani/fer-cnn.git'  #@param {type:"string"}
BRANCH = 'main'  #@param {type:"string"}

!git clone --branch {BRANCH} {REPO_URL} fer-cnn
%cd fer-cnn

In [ ]:
!pip install -q -r requirements.txt

## 2. Datasets

Enter your Kaggle username and API key, from `kaggle.com > Settings > API`.

In [ ]:
import getpass
import json
import os

kaggle_username = input('Kaggle username: ')
kaggle_key = getpass.getpass('Kaggle API key: ')

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump({'username': kaggle_username, 'key': kaggle_key}, f)
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

!pip install -q kaggle

In [ ]:
!mkdir -p data/raw/fer-2013 data/raw/fane_data
!kaggle datasets download -d msambare/fer2013 -p data/raw/fer-2013 --unzip
!kaggle datasets download -d furcifer/fane-facial-expressions-and-emotion-dataset -p data/raw/fane_data --unzip

## 3. Drive and helpers

Every experiment writes to the usual project folders, then gets copied into its
own subfolder on Drive so the next one cannot overwrite it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

DRIVE_DIR = '/content/drive/MyDrive/fer-cnn-results'  #@param {type:"string"}
EPOCHS = 100  #@param {type:"integer"}
WARMUP_EPOCHS = 5  #@param {type:"integer"}

DRIVE_ROOT = Path(DRIVE_DIR)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)


def run(*args):
    """Runs one of the project scripts, streaming its output into this cell.

    subprocess.run() alone writes straight to the kernel's file descriptors, so
    on Colab the output never reaches the cell and a failure shows up as a bare
    exit code with no traceback. Reading the pipe line by line fixes that.
    """
    print('$ python', *args, flush=True)
    process = subprocess.Popen(
        [sys.executable, *args],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end='')
    code = process.wait()
    if code != 0:
        raise RuntimeError(f'{args[0]} failed with exit code {code}, see the output above')


def archive(name, models=()):
    """Copies results/ and the given checkpoints into DRIVE_ROOT/name/."""
    dest = DRIVE_ROOT / name
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copytree('results', dest / 'results', dirs_exist_ok=True)
    for model in models:
        checkpoint = Path('models_saved') / f'{model}.keras'
        if checkpoint.exists():
            shutil.copy2(checkpoint, dest / f'{model}.keras')
    print(f'saved to {dest}')


def experiment(name, model, warmup=0, epochs=EPOCHS):
    """Trains one model, evaluates it, redraws the plots and archives everything."""
    run('src/train.py', '--model', model, '--epochs', str(epochs),
        '--warmup_epochs', str(warmup))
    run('src/evaluate.py', '--models', model)
    run('src/plot_results.py')
    archive(name, models=(model,))

## 4. Exploratory analysis

Class counts and sample grids. Runs in under a minute and needs no GPU.

In [ ]:
run('src/eda.py')
archive('eda')

## 5. Experiments

One cell each, so a single failure does not cost the whole session.

### 5.1 Custom CNN, trained from scratch

In [ ]:
experiment('custom_cnn', 'custom_cnn')

### 5.2 Xception, single phase

The tail is unfrozen from the first epoch, with the head still random. This is
the configuration used in the first version of the study.

In [ ]:
experiment('xception_1phase', 'xception', warmup=0)

### 5.3 Xception, two phases

The base stays frozen for the first `WARMUP_EPOCHS` epochs so the head can
settle, then `block14` is unfrozen and training continues at the lower rate.

In [ ]:
experiment('xception_2phase', 'xception', warmup=WARMUP_EPOCHS)

## 6. Final comparison

Picks the better Xception variant on validation loss, then evaluates both models
together to produce the tables and figures used in the report.

In [ ]:
import json


def best_val_loss(name):
    path = DRIVE_ROOT / name / 'results' / 'metrics' / 'xception_history.json'
    return min(json.loads(path.read_text())['val_loss'])


variants = {name: best_val_loss(name) for name in ('xception_1phase', 'xception_2phase')}
for name, value in variants.items():
    print(f'{name:<16} best val_loss {value:.4f}')

# selection happens on validation only, the test sets play no part in it
winner = min(variants, key=variants.get)
print(f'\nselected variant: {winner}')

shutil.copy2(DRIVE_ROOT / winner / 'xception.keras', 'models_saved/xception.keras')
shutil.copy2(DRIVE_ROOT / winner / 'results' / 'metrics' / 'xception_history.json',
             'results/metrics/xception_history.json')

run('src/evaluate.py')
run('src/plot_results.py')
archive('final', models=('custom_cnn', 'xception'))

## 7. Results

In [ ]:
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv('results/metrics/comparison.csv'))
display(pd.read_csv('results/metrics/generalization_gap.csv'))

for figure in ('training_curves.png', 'generalization_gap_slope.png'):
    display(Image(f'results/figures/{figure}'))